In [1]:
!pip install gspread
!pip install gspread-dataframe

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.colab import auth
from google.auth import default

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt

In [3]:
from google.colab import drive
drive.mount('/content/drive')

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

planilha = gc.open('alunos_detalhes')
aba = planilha.sheet1

dados = aba.get_all_values()
df = pd.DataFrame(dados[1:], columns=dados[0])

Mounted at /content/drive


Limpeza e conversão da coluna salario_medio e coluna nota

In [8]:
df['salario_medio'] = df['salario_medio'].replace('', np.nan)
df['salario_medio'] = pd.to_numeric(df['salario_medio'], errors='coerce')
print(df['salario_medio'].dtype)

df['nota'] = df['nota'].replace('', np.nan)
df['nota'] = pd.to_numeric(df['nota'], errors='coerce')
print(df['nota'].dtype)

float64
float64


Remover linhas com NaN em 'salario_medio' e 'nota'

In [9]:
df = df.dropna(subset=['salario_medio', 'nota'])

Pré processamento dos dados, removendo iformações  irrelevantes

In [10]:
x = df.drop(["aluno_id", "nome_aluno", "email_aluno", "data_nascimento"], axis=1)
y = df["nota"]

Garantir que X não possua valores nulos

In [11]:
x = x.dropna()

Separando colunas numericas e categóricas

In [12]:
numeric_features = ['salario_medio']
categorical_features = ['curso', 'turma', 'trabalha', 'cidade', 'estado',
                         'uso_alcool', 'fuma', 'uso_drogas', 'problemas_mentais', 'materia',
                         'professor']

Criando um transformers para os pré-processamentos

In [13]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

Combinando transformers em um ColumnTransformer

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

Divisão de dados de treinamento e testes

In [15]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

Garantir que X_train seja um DataFrame com as colunas corretas

In [16]:
X_train = pd.DataFrame(X_train, columns=x.columns)
X_test = pd.DataFrame(X_test, columns=x.columns)

Garantir que as colunas numéricas em X_train e X_test sejam do tipo correto

In [17]:
for col in numeric_features:
    if col in X_train.columns:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    if col in X_test.columns:
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

Seleção do modelo

In [18]:
model = RandomForestRegressor(n_estimators=100, random_state=42)

Criando a Pipeline completa

In [19]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model)
])

Treinamento do Modelo

In [20]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['salario_medio']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['curso', 'turma', 'trabalha',
                                                   'cidade', 'estado',
                                                   'uso_alcool', 'fuma',
                                                   'uso_drogas',
                                                   'problemas_mentais',
                                                   'materia', 'professor'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

Avaliação do Modelo

In [21]:
y_pred = pipeline.predict(X_test)

Métricas de Classificação

In [22]:
print(f'Acurácia: {accuracy_score(y_test, y_pred)}')
print(f'Precisão: {precision_score(y_test, y_pred, average='weighted', zero_division=1)}')
print(f'Recall: {recall_score(y_test, y_pred, average='weighted')}')
print(f'F1-Score: {f1_score(y_test, y_pred, average='weighted', zero_division=1)}')

SyntaxError: f-string: unmatched '(' (<ipython-input-22-c62eced3e203>, line 2)

Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)
classes = sorted(df['nota'].unique())

Plotando a Matriz de Confusão

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusão')
plt.xlabel('Previsão')
plt.ylabel('Verdadeiro')
plt.show()

KeyboardInterrupt: 

Error in callback <function _draw_all_if_interactive at 0x7b91d90f6700> (for post_execute):


KeyboardInterrupt: 

Error in callback <function flush_figures at 0x7b91d90f4040> (for post_execute):


KeyboardInterrupt: 